In [ ]:
# Add repo root to Python path
import sys
from pathlib import Path

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"✅ Added to path: {repo_root}")

## Import modules

In [ ]:
from src.embedding.embedder import HuggingFaceEmbedder, normalize_embeddings
from src.ingestion.loaders import DataLoader
from src.config import Config
import numpy as np

## Load chunks

In [ ]:
# Load chunks from previous step
chunks_path = Config.MVP_KB_OUTPUTS / "mdn_chunks.jsonl"
chunks = DataLoader.load_jsonl(chunks_path)

print(f"📦 Loaded chunks: {len(chunks)}")
print(f"Sample keys: {list(chunks[0].keys())}")

## Initialize embedding provider

In [ ]:
# Initialize HuggingFace embedder with BGE-small model
embedder = HuggingFaceEmbedder(
    model="BAAI/bge-small-en-v1.5"
)

print(f"✅ Embedding provider initialized")
print(f"Model: {embedder.model}")
print(f"Query prefix: {embedder.query_prefix[:50]}..." if embedder.query_prefix else "No prefix")

## Test embedding on single text

In [ ]:
# Test single embedding
test_text = "Semantic HTML provides meaning to web content"
test_emb = embedder.embed_query(test_text)

print(f"\nTest embedding:")
print(f"  Input: {test_text}")
print(f"  Embedding shape: {test_emb.shape}")
print(f"  First 5 values: {test_emb[:5]}")

## Generate embeddings for all chunks (batched)

In [ ]:
# Extract chunk texts
chunk_texts = [c["text"] for c in chunks]

print(f"\n📝 Preparing to embed {len(chunk_texts)} chunks...")
print(f"This may take a few minutes...")

In [ ]:
# Generate embeddings in batches
chunk_embeddings = embedder.embed_in_batches(
    chunk_texts,
    batch_size=16,
    is_query=False,
    verbose=True
)

print(f"\n✅ Generated embeddings")
print(f"Shape: {chunk_embeddings.shape}")
print(f"Dtype: {chunk_embeddings.dtype}")

## Save embeddings

In [ ]:
# Save to numpy file
embeddings_path = Config.MVP_KB_OUTPUTS / "chunk_embeddings.npy"
np.save(embeddings_path, chunk_embeddings)

print(f"✅ Saved embeddings to: {embeddings_path}")
print(f"File size: {embeddings_path.stat().st_size / 1024 / 1024:.2f} MB")

## Test semantic search

In [ ]:
# Normalize embeddings for cosine similarity
chunk_embeddings_norm = normalize_embeddings(chunk_embeddings)

# Test query
query = "Explain semantic HTML"
query_vec = embedder.embed_query(query)
query_vec_norm = normalize_embeddings(query_vec)

# Compute similarities
similarities = chunk_embeddings_norm @ query_vec_norm

# Get top 5 results
top_k = 5
top_indices = np.argsort(similarities)[-top_k:][::-1]

print(f"\n🔍 Query: '{query}'")
print(f"\nTop {top_k} results:")
for i, idx in enumerate(top_indices, 1):
    chunk = chunks[idx]
    score = similarities[idx]
    print(f"\n{i}. Score: {score:.4f}")
    print(f"   Title: {chunk['title']}")
    print(f"   Heading: {chunk['heading']}")
    print(f"   Text preview: {chunk['text'][:150]}...")

---

## Summary

**What this demo does:**
1. Loads chunks from Step 02
2. Initializes HuggingFace embedding provider (BGE-small)
3. Generates embeddings in batches
4. Saves embeddings for future use
5. Demonstrates semantic search

**Next step:** Store in Pinecone vector DB (see `demo_08_pinecone_ltm.ipynb`)